# Solar proton event forecasting with IBM Granite (TTM)

Forecasts the onset of solar radiation storms (NOAA S1: ≥10 pfu of ≥10 MeV protons) from the OMNI 5-min dataset, and benchmarks IBM Granite TinyTimeMixer against baselines.

| Step | What it does | Time (approx.) |
|---|---|---|
| 1 | Install `granite-tsfm` | 1 min |
| 2 | Load the CSV (upload or Google Drive) | 1–5 min |
| 3 | Baselines: persistence + gradient boosting, two evaluation splits | 2–4 min |
| 4 | Granite zero-shot | GPU ~1 min / CPU ~5–10 min |
| 5 | Granite fine-tuned (head only, trained on ≤2013) | GPU ~2 min / CPU ~10+ min |
| 6 | Comparison table on the 2014–2020 test set | instant |

**Tip:** Runtime → Change runtime type → T4 GPU makes steps 4–5 much faster. Everything also runs on CPU.

**Task:** at each 15-min time step with flux below 10 pfu and no storm in the last 24 h, will a new S1 storm start within 1, 6 or 24 h? Proton data in the file covers 2010 – Mar 2020 (41 storms).

## 1. Install
granite-tsfm pins PyTorch < 2.12, so pip downgrades Colab's PyTorch. Colab's torchvision then no longer matches and can crash the Granite import, so it is removed (this project doesn't use it). The dependency-conflict warnings pip prints here are expected.

**After this cell finishes: Runtime → Restart session, then run from step 2 onward.**

In [ ]:
!pip install -q granite-tsfm
!pip uninstall -y -q torchvision torchaudio
!python -c "import torch, tsfm_public; print('granite-tsfm OK | torch', torch.__version__, '| GPU:', torch.cuda.is_available())"

## 2. Load the data
The CSV is ~140 MB. Uploading through the browser works but is slow; putting it in Google Drive and setting `USE_DRIVE = True` is faster.

In [ ]:
import os
CSV = "nasa_omni_solar_wind_storms_5min_2010-2026.csv"
USE_DRIVE = False                      # True: read from Google Drive instead of uploading
DRIVE_PATH = "/content/drive/MyDrive/" + CSV   # edit if the file is in a subfolder

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CSV = DRIVE_PATH
elif not os.path.exists(CSV):
    from google.colab import files
    up = files.upload()                # pick the CSV
    CSV = next(iter(up))
print("using:", CSV, "| exists:", os.path.exists(CSV))

## 3. Pipeline code
These two cells write the scripts to disk. `proton_forecast.py` = data, labels, baselines and evaluation. `granite_ttm.py` = the Granite add-on, scored on exactly the same rows.

In [ ]:
%%writefile proton_forecast.py
#!/usr/bin/env python3
"""
Solar energetic proton (>=10 MeV) onset forecasting from the OMNI 5-min file.

Task   : at time t (flux currently < 10 pfu and no S1 in the previous 24 h),
         will a NEW S1 event (>=10 pfu) begin within the next H hours? H = 1, 6, 24.
Models : persistence (score = current flux), HGB on flux-history features,
         HGB on flux + solar-wind features. Climatology = base rate (Brier reference).
Eval   : (a) leave-one-year-out, whole-year blocks with a 3-day purge on each side,
         (b) strict forward split: train <= 2013, test >= 2014.
         Sample metrics (AUC, PR-AUC, Brier skill) + event metrics at a fixed 1% false-positive
         rate (event detection, lead time, alarm episodes/month, episode precision).

Usage: python3 -I proton_forecast.py OMNI.csv OUT_DIR
Any other forecaster (e.g. IBM Granite) can be scored by adding a column to `scores` in run_split.
"""
import sys, os, time
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

S1 = 10.0          # pfu, NOAA S1 threshold (>=10 MeV)
QUIET = 288        # 24 h of 5-min steps
STEP = 3           # forecast every 15 min
HORIZONS = [1, 6, 24]
PURGE = pd.Timedelta(days=3)
FPR = 0.01

# ---------------------------------------------------------------- data
def load(path):
    df = pd.read_csv(path, parse_dates=["datetime_utc"]).set_index("datetime_utc")
    return df.loc[: df.proton_flux_10mev.last_valid_index()]       # proton data end 2020-03

def csum_window(flag, lo, hi):
    """sum of flag over steps (t+lo .. t+hi) inclusive, clipped to array bounds"""
    c = np.concatenate([[0], np.cumsum(flag.astype(np.int64))])
    n = len(flag); t = np.arange(n)
    a = np.clip(t + lo, 0, n); b = np.clip(t + hi + 1, 0, n)
    return c[b] - c[a]

def features(df):
    p = df.proton_flux_10mev
    lp = np.log10(p)
    F = pd.DataFrame(index=df.index)
    F["lp"] = lp
    for w in (3, 12, 72, 288):
        F[f"lp_max_{w}"] = lp.rolling(w, min_periods=1).max()
    for d in (3, 12, 36, 72):
        F[f"lp_d{d}"] = lp - lp.shift(d)
    F["lp_acc"] = (lp - lp.shift(3)) - (lp.shift(3) - lp.shift(6))
    flux_cols = list(F.columns)
    sw = pd.DataFrame(index=df.index)
    for c in ["speed_km_s", "density_cm3", "pressure_nPa", "bt_nT", "bz_gsm_nT", "efield_mV_m", "sym_h_nT", "ae_nT"]:
        sw[c] = df[c]
    sw["speed_mean_72"] = df.speed_km_s.rolling(72, min_periods=12).mean()
    sw["speed_max_288"] = df.speed_km_s.rolling(288, min_periods=36).max()
    sw["speed_d72"] = df.speed_km_s - df.speed_km_s.shift(72)
    sw["dens_max_72"] = df.density_cm3.rolling(72, min_periods=12).max()
    sw["bt_max_288"] = df.bt_nT.rolling(288, min_periods=36).max()
    sw["symh_min_288"] = df.sym_h_nT.rolling(288, min_periods=36).min()
    sw["ae_max_72"] = df.ae_nT.rolling(72, min_periods=12).max()
    return pd.concat([F, sw], axis=1), flux_cols

def labels_and_events(df):
    p = df.proton_flux_10mev.values
    valid = ~np.isnan(p)
    cross = np.where(valid, p >= S1, False)
    prior = csum_window(cross, -QUIET, -1)                       # S1 samples in previous 24 h
    onset = cross & (prior == 0)
    eligible = valid & (p < S1) & (csum_window(cross, -QUIET, 0) == 0)
    Y, OK = {}, {}
    for H in HORIZONS:
        n = H * 12
        Y[H] = csum_window(cross, 1, n) > 0
        OK[H] = csum_window(valid, 1, n) >= 0.8 * n              # >=80% of future window observed (or crossing seen)
        OK[H] |= Y[H]
    oi = np.flatnonzero(onset)
    bounds = np.append(oi, len(p))
    ev = pd.DataFrame({"onset": df.index[oi]})
    ev["peak_pfu"] = [np.nanmax(p[bounds[i]:bounds[i + 1]]) for i in range(len(oi))]
    ev["sev"] = pd.cut(ev.peak_pfu, [10, 100, 1000, 1e9], right=False, labels=["S1", "S2", "S3+"])
    return Y, OK, eligible, ev

# ---------------------------------------------------------------- evaluation
def event_metrics(t, y, score, onsets, sev, H):
    """fixed-FPR operating point; returns dict + per-event table"""
    thr = np.quantile(score[~y], 1 - FPR)
    alarm = score >= thr
    Hd = pd.Timedelta(hours=H)
    rows = []
    for o, s in zip(onsets, sev):
        m = (t >= o - Hd) & (t < o)
        if not m.any():
            continue                                             # onset not forecastable (e.g. ongoing storm tail)
        hit = m & alarm
        lead = (o - t[hit].min()).total_seconds() / 3600 if hit.any() else np.nan
        rows.append((o, s, bool(hit.any()), lead))
    E = pd.DataFrame(rows, columns=["onset", "sev", "detected", "lead_h"])
    # alarm episodes
    ta = t[alarm]
    ep_id = np.cumsum(np.r_[True, np.diff(ta.values).astype("timedelta64[m]").astype(int) > 60]) if len(ta) else np.array([])
    n_ep, true_ep = 0, 0
    for e in np.unique(ep_id):
        te = ta[ep_id == e]
        n_ep += 1
        true_ep += bool(((onsets >= te.min()) & (onsets <= te.max() + Hd)).any())
    months = len(t) * STEP * 5 / 60 / 24 / 30.4
    out = dict(thr=thr, n_events=len(E), POD=E.detected.mean() if len(E) else np.nan,
               median_lead_h=E.lead_h.median(), lead_ge_1h=(E.lead_h >= 1).mean() if len(E) else np.nan,
               episodes_per_month=n_ep / months, episode_precision=true_ep / n_ep if n_ep else np.nan)
    return out, E

def hgb():
    return HistGradientBoostingClassifier(max_iter=150, learning_rate=0.06, max_depth=3,
                                          min_samples_leaf=100, l2_regularization=1.0, random_state=0)

def run_split(name, X, flux_cols, Y, OK, elig, test_mask_fn, train_mask_fn, onsets, sev, idx_all, times):
    """test_mask_fn(fold) / train_mask_fn(fold) -> boolean over idx_all; folds iterate"""
    res_rows, ev_tabs = [], []
    for H in HORIZONS:
        base = elig[idx_all] & OK[H][idx_all]
        preds = {k: np.full(len(idx_all), np.nan) for k in ["persistence", "flux_only", "flux+solar_wind"]}
        clim = np.full(len(idx_all), np.nan)
        for fold in test_mask_fn.folds:
            te = base & test_mask_fn(fold)
            tr = base & train_mask_fn(fold)
            ytr = Y[H][idx_all][tr]
            clim[te] = ytr.mean()
            preds["persistence"][te] = X["lp"].values[idx_all][te]
            for key, cols in [("flux_only", flux_cols), ("flux+solar_wind", list(X.columns))]:
                m = hgb().fit(X[cols].values[idx_all][tr], ytr)
                preds[key][te] = m.predict_proba(X[cols].values[idx_all][te])[:, 1]
        sel = ~np.isnan(clim)
        y = Y[H][idx_all][sel]; t = times[idx_all][sel]
        for key, sc in preds.items():
            s = sc[sel]; ok = ~np.isnan(s)
            auc = roc_auc_score(y[ok], s[ok]); ap = average_precision_score(y[ok], s[ok])
            bss = np.nan
            if key != "persistence":
                bs = np.mean((s[ok] - y[ok]) ** 2); bc = np.mean((clim[sel][ok] - y[ok]) ** 2)
                bss = 1 - bs / bc
            em, E = event_metrics(t[ok], y[ok], s[ok], onsets, sev, H)
            res_rows.append(dict(split=name, H_h=H, model=key, n_samples=int(ok.sum()), n_pos=int(y[ok].sum()),
                                 AUC=auc, PR_AUC=ap, base_rate=y[ok].mean(), Brier_skill=bss, **em))
            E.insert(0, "model", key); E.insert(0, "H_h", H); E.insert(0, "split", name)
            ev_tabs.append(E)
    return pd.DataFrame(res_rows), pd.concat(ev_tabs)

class Folds:
    def __init__(self, folds, fn):
        self.folds, self._fn = folds, fn
    def __call__(self, fold):
        return self._fn(fold)

def main(path, out):
    os.makedirs(out, exist_ok=True)
    t0 = time.time()
    df = load(path)
    X, flux_cols = features(df)
    Y, OK, elig, ev = labels_and_events(df)
    ev.to_csv(f"{out}/events.csv", index=False)
    print(f"{len(df):,} rows {df.index[0]} -> {df.index[-1]};  events: {ev.sev.value_counts().to_dict()}")
    idx_all = np.arange(0, len(df), STEP)
    times = df.index
    yr = times.year.values[idx_all]
    tt = times[idx_all]
    onsets, sev = pd.DatetimeIndex(ev.onset), ev.sev.values

    # (a) leave-one-year-out, 2018-2020 pooled (no events, only negatives)
    groups = [(y, y) for y in range(2010, 2018)] + [(2018, 2020)]
    def span(g):
        return pd.Timestamp(f"{g[0]}-01-01"), pd.Timestamp(f"{g[1]}-12-31 23:59")
    def te_fn(g):
        a, b = span(g); return (tt >= a) & (tt <= b)
    def tr_fn(g):
        a, b = span(g); return (tt < a - PURGE) | (tt > b + PURGE)
    loyo = Folds(groups, te_fn); loyo_tr = Folds(groups, tr_fn)
    r1, e1 = run_split("leave-one-year-out", X, flux_cols, Y, OK, elig, loyo, loyo_tr, onsets, sev, idx_all, times)
    print(f"LOYO done {time.time()-t0:.0f}s")

    # (b) strict forward split
    cut = pd.Timestamp("2014-01-01")
    fw = Folds(["fwd"], lambda g: tt >= cut); fw_tr = Folds(["fwd"], lambda g: tt < cut - PURGE)
    r2, e2 = run_split("forward (train<=2013, test>=2014)", X, flux_cols, Y, OK, elig, fw, fw_tr,
                       onsets[onsets >= cut], sev[onsets >= cut], idx_all, times)
    R = pd.concat([r1, r2]); E = pd.concat([e1, e2])
    R.to_csv(f"{out}/results.csv", index=False); E.to_csv(f"{out}/per_event.csv", index=False)
    pd.set_option("display.width", 250, "display.float_format", lambda v: f"{v:.3f}")
    print(R.drop(columns=["thr"]).to_string(index=False))
    # detection by severity at the fixed-FPR point
    S = (E.groupby(["split", "H_h", "model", "sev"], observed=True).detected.agg(["mean", "count"]).reset_index())
    S.to_csv(f"{out}/pod_by_severity.csv", index=False)
    print(S[S.H_h == 6].to_string(index=False))
    print(f"total {time.time()-t0:.0f}s")

if __name__ == "__main__":
    main(sys.argv[1], sys.argv[2])

In [ ]:
%%writefile granite_ttm.py
#!/usr/bin/env python3
"""
IBM Granite TinyTimeMixer (TTM r2) for solar proton event forecasting.
Plugs into proton_forecast.py: same data, labels, eligibility, forward split
(train <= 2013, test >= 2014) and event metrics, so results line up with results.csv.

How Granite is used
  1. Forecast log10(>=10 MeV proton flux) on a 15-min grid:
     context 512 steps (5.3 days) -> prediction 96 steps (24 h).
  2. Event score for horizon H = max forecast log-flux over the next H hours
     ("does the forecast cross 10 pfu?"). Scored exactly like persistence.

Modes
  default      zero-shot: pretrained TTM, no training
  --finetune   fine-tune the TTM head on training years (<= 2013), backbone frozen
  --smoke      random-init TTM on a small subset; checks the plumbing only (no download)

Install : pip install granite-tsfm          (pulls torch + transformers)
Run     : python granite_ttm.py OMNI.csv OUT_DIR [--finetune]
Needs proton_forecast.py in the same folder. CPU is fine (TTM is ~1M parameters).
"""
import sys, os, argparse, time
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np, pandas as pd, torch
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.metrics import roc_auc_score, average_precision_score
import proton_forecast as pf

CTX, PRED = 512, 96                       # 15-min steps
DEV = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "ibm-granite/granite-timeseries-ttm-r2"
CUT = pd.Timestamp("2014-01-01")
LOG_S1 = np.log10(pf.S1)


def load_ttm(smoke):
    if smoke:
        from tsfm_public import TinyTimeMixerConfig, TinyTimeMixerForPrediction
        cfg = TinyTimeMixerConfig(context_length=CTX, prediction_length=PRED, num_input_channels=1,
                                  patch_length=64, patch_stride=64)
        return TinyTimeMixerForPrediction(cfg)
    try:
        from tsfm_public.toolkit.get_model import get_model
        return get_model(MODEL_ID, context_length=CTX, prediction_length=PRED)
    except Exception as e:                # older granite-tsfm: weights live on a revision branch
        print("get_model failed, using revision branch:", e)
        from tsfm_public import TinyTimeMixerForPrediction
        return TinyTimeMixerForPrediction.from_pretrained(MODEL_ID, revision="512-96-r2")


@torch.no_grad()
def forecast(model, s, origins, bs=512):
    """origins: indices k on the 15-min grid; context = s[k-CTX+1 .. k]; returns [n, PRED]"""
    W = sliding_window_view(s, CTX)
    out = np.empty((len(origins), PRED), np.float32)
    model.eval()
    for i in range(0, len(origins), bs):
        k = origins[i:i + bs]
        x = torch.from_numpy(W[k - CTX + 1].copy()).unsqueeze(-1).to(DEV)   # [B, CTX, 1]
        out[i:i + bs] = model(past_values=x).prediction_outputs[:, :PRED, 0].cpu().numpy()
    return out


def finetune(model, s, tt, epochs=3, n=40000, lr=1e-3, bs=256, seed=0):
    """Train the head on <=2013 windows; half of each epoch drawn from windows whose
    next 24 h reach >=1 pfu, so the rare ramps are not drowned out by quiet time."""
    rng = np.random.default_rng(seed)
    W, Wf = sliding_window_view(s, CTX), sliding_window_view(s, PRED)
    last = np.flatnonzero(tt < CUT - pf.PURGE).max() - PRED
    k_all = np.arange(CTX - 1, last)
    active = Wf[k_all + 1].max(1) >= 0.0
    k_act, k_quiet = k_all[active], k_all[~active]
    for p in model.backbone.parameters():
        p.requires_grad = False
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    model.train()
    for ep in range(epochs):
        k = np.concatenate([rng.choice(k_act, n // 2), rng.choice(k_quiet, n // 2)])
        rng.shuffle(k)
        tot = 0.0
        for i in range(0, len(k), bs):
            kk = k[i:i + bs]
            x = torch.from_numpy(W[kk - CTX + 1].copy()).unsqueeze(-1).to(DEV)
            y = torch.from_numpy(Wf[kk + 1].copy()).unsqueeze(-1).to(DEV)
            loss = model(past_values=x, future_values=y).loss
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(kk)
        print(f"  epoch {ep + 1}/{epochs}  train MSE {tot / len(k):.4f}")
    return model


def export_replay(df, ev, tt, test_k, F, thr6, tag, path, before="2.5D", after="1.5D"):
    """Replay file for the console page: for each test storm, 5-min observed log10 flux over
    [onset - before, onset + after] and, if F is given, the model's 96-step (24 h) forecast made at
    every 15-min time in that window (forecast k covers origin + 15 min * (1..96))."""
    import json
    p = df.proton_flux_10mev
    row = {k: r for r, k in enumerate(test_k)}
    kpos = pd.Series(np.arange(len(tt)), index=tt)

    def window(t0, t1):
        fl = np.log10(p.loc[t0:t1 - pd.Timedelta("5min")].values)
        item = dict(t0=str(t0), flux=[None if np.isnan(v) else round(float(v), 5) for v in fl])
        if F is not None:
            ks = kpos.loc[t0:t1 - pd.Timedelta("15min")].values
            item["fc"] = [[round(float(v), 2) for v in F[row[k]]] if k in row else None for k in ks]
            if thr6 is not None:   # exact 6 h alarm decision per forecast (rounded forecasts can flip it)
                item["g6"] = "".join("1" if (k in row and F[row[k], :24].max() >= thr6) else "0" for k in ks)
        return item

    storms = []
    for o, pk, sv in zip(ev.onset, ev.peak_pfu, ev.sev):
        if o < CUT:
            continue
        t0 = (o - pd.Timedelta(before)).floor("15min"); t1 = (o + pd.Timedelta(after)).floor("15min")
        storms.append(dict(kind="storm", onset=str(o), peak_pfu=round(float(pk), 1), sev=str(sv), **window(t0, t1)))

    # every watch (yellow) that never became a storm, using the same tier rules as tier_alerts.py
    watches = []
    if F is not None and thr6 is not None:
        import tier_alerts as ta
        q = p.loc[CUT:]; t = q.index; x = q.values
        m6 = pd.Series(F[:, :24].max(1), index=tt[test_k])
        g = (m6.reindex(t, method="ffill", limit=2) >= thr6).values         # NaN compares False
        tr, trO = ta.tiers(x, g), ta.tiers(x)
        E = ta.yellow_episodes(t, tr)
        for e in E[~E.reached_orange].itertuples():
            si = t.get_loc(e.start)
            t0 = (e.start - pd.Timedelta("18h")).floor("15min")
            t1 = min(e.end + pd.Timedelta("18h"), e.start + pd.Timedelta("3D")).floor("15min")
            watches.append(dict(kind="watch", start=str(e.start), end=str(e.end),
                                raised_by="READYation" if trO[si] == 0 else "flux",
                                duration_h=round((e.end - e.start).total_seconds() / 3600 + 5 / 60, 2),
                                peak_pfu=round(float(np.nanmax(q.loc[e.start:e.end].values)), 2),
                                **window(t0, t1)))
        print(f"watches that did not become storms: {len(watches)} "
              f"({sum(w['raised_by'] == 'READYation' for w in watches)} raised by READYation first)")
    out = dict(model=tag, thr_6h_log10=None if thr6 is None else round(float(thr6), 4), step_flux_min=5,
               step_fc_min=15, fc_steps=PRED, storms=storms, watches=watches)
    with open(path, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print(f"saved replay file -> {path}  ({len(storms)} storms, {len(watches)} no-storm watches, "
          f"{os.path.getsize(path) / 1e6:.1f} MB)")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("csv"); ap.add_argument("out")
    ap.add_argument("--finetune", action="store_true")
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--replay", action="store_true", help="also write out/readyation_replay.json for the console page")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True); t0 = time.time()
    torch.manual_seed(0)

    df = pf.load(a.csv)
    Y, OK, elig, ev = pf.labels_and_events(df)
    idx = np.arange(0, len(df), pf.STEP)                  # 15-min grid, same as proton_forecast.py
    tt = df.index[idx]
    lp = np.log10(df.proton_flux_10mev).iloc[idx]
    s = lp.ffill().fillna(lp.median()).values.astype(np.float32)   # past-only fill, no leakage

    test_k = np.flatnonzero(tt >= CUT)                    # forecast every test time (needed for the alert tiers);
    if a.smoke:                                           # evaluation below still uses only eligible times
        test_k = test_k[::50]
    el = elig[idx][test_k]
    onsets = pd.DatetimeIndex(ev.onset); sev = ev.sev.values
    keep = onsets >= CUT; onsets, sev = onsets[keep], sev[keep]

    model = load_ttm(a.smoke).to(DEV)
    print("device:", DEV)
    tag = "granite_ttm_zeroshot"
    if a.finetune:
        model = finetune(model, s, tt, **(dict(epochs=1, n=1024) if a.smoke else {}))
        tag = "granite_ttm_finetuned"
    print(f"forecasting {len(test_k):,} origins ...")
    F = forecast(model, s, test_k)

    rows = []
    for H in pf.HORIZONS:
        m = OK[H][idx][test_k] & el
        y = Y[H][idx][test_k][m]; t = tt[test_k][m]
        scores = {"persistence": s[test_k][m], tag: F[m, :H * 4].max(1)}
        for name, sc in scores.items():
            em, _ = pf.event_metrics(t, y, sc, onsets, sev, H)
            rows.append(dict(split="forward (train<=2013, test>=2014)", H_h=H, model=name,
                             n_samples=int(m.sum()), n_pos=int(y.sum()),
                             AUC=roc_auc_score(y, sc), PR_AUC=average_precision_score(y, sc),
                             forecast_crosses_S1=float((sc >= LOG_S1).mean()) if name != "persistence" else np.nan,
                             **em))
    R = pd.DataFrame(rows)
    R.to_csv(f"{a.out}/{tag}_results.csv", index=False)
    # per-time forecasts for tier_alerts.py: max forecast log10 flux over the next 1/6/24 h,
    # plus the 6 h alarm threshold (1% false-positive point) used for the YELLOW tier
    thr6 = R[(R.model == tag) & (R.H_h == 6)].thr.item()
    pd.DataFrame({"time": tt[test_k], "max_lp_1h": F[:, :4].max(1), "max_lp_6h": F[:, :24].max(1),
                  "max_lp_24h": F[:, :96].max(1), "thr_6h": thr6}).to_csv(f"{a.out}/{tag}_scores.csv.gz", index=False)
    print(f"saved per-time forecasts -> {a.out}/{tag}_scores.csv.gz")
    if a.replay:
        export_replay(df, ev, tt, test_k, F, thr6, tag, f"{a.out}/readyation_replay.json")
    pd.set_option("display.width", 250, "display.float_format", lambda v: f"{v:.3f}")
    print(R.drop(columns=["thr"]).to_string(index=False))
    print(f"total {time.time() - t0:.0f}s" + ("   [SMOKE TEST: random weights, numbers meaningless]" if a.smoke else ""))


if __name__ == "__main__":
    main()

## 4. Baselines
Persistence (score = current flux), gradient boosting on flux history, and gradient boosting with solar wind. Evaluated two ways: leave-one-year-out, and a strict forward split (train ≤2013, test ≥2014). Writes `out/results.csv`, `out/per_event.csv`, `out/events.csv`.

In [ ]:
!python proton_forecast.py "{CSV}" out

## 5. IBM Granite TTM
Forecasts the next 24 h of log proton flux from the previous 5.3 days (15-min steps). Event score for horizon H = highest forecast flux in the next H hours. The first run downloads the pretrained weights from Hugging Face (a warning about `HF_TOKEN` is harmless).

In [ ]:
!python granite_ttm.py "{CSV}" out              # zero-shot

In [ ]:
!python granite_ttm.py "{CSV}" out --finetune   # fine-tune head on 2010–2013, backbone frozen

## 6. Compare on the 2014–2020 test set
All rows use the same forecast times, labels and alarm rule (alarms on 1% of quiet time). Persistence is the bar to beat.

- **AUC / PR_AUC**: ranking quality per 15-min forecast (PR_AUC matters more, since storms are rare)
- **POD**: share of storms with an alarm before onset
- **median_lead_h**: warning time before onset
- **episodes_per_month / episode_precision**: how often alarms fire, and the share followed by a real storm

In [ ]:
import pandas as pd
fwd = "forward (train<=2013, test>=2014)"
parts = [pd.read_csv("out/results.csv").query("split == @fwd")]
for f in ["granite_ttm_zeroshot_results.csv", "granite_ttm_finetuned_results.csv"]:
    if os.path.exists(f"out/{f}"):
        parts.append(pd.read_csv(f"out/{f}"))
R = pd.concat(parts).drop_duplicates(subset=["H_h", "model"])   # persistence appears in each file
cols = ["H_h", "model", "n_samples", "n_events", "AUC", "PR_AUC", "POD", "median_lead_h",
        "episodes_per_month", "episode_precision"]
R[cols].sort_values(["H_h", "PR_AUC"], ascending=[True, False]).round(3)

### Reading the result
- Granite **beats persistence** (higher PR_AUC, similar POD with fewer alarm episodes or longer lead) → a real finding for the write-up.
- Granite **≈ persistence** → it has learned to extrapolate the flux rise; the signal is the early ramp itself, and earlier warning needs flare/CME inputs (GOES X-ray, NASA DONKI).
- With 14 storms in the test set, small differences are noise; compare across horizons, not single numbers.

## 7. Four-tier radiation alerts
| Tier | Rule | Meaning |
|---|---|---|
| 🟢 GREEN | quiet | — |
| 🟡 YELLOW | observed flux ≥1 pfu **or Granite's 6 h forecast** crosses its alarm level (on after 15 min, off after 2 h without trigger) | watch: storm likely within hours |
| 🟠 ORANGE | observed flux ≥10 pfu (NOAA S1–S2) | storm: heightened monitoring, review EVA plans |
| 🔴 RED | observed flux ≥1000 pfu (NOAA S3+) | EVA hazard (NOAA: avoidance recommended for astronauts on EVA) |

Run once with observed flux only (baseline) and once with fine-tuned Granite raising yellow. The Granite run also checks, as an exploratory result, whether its 24 h forecast saw RED coming (only 2 RED events in the test period).

In [ ]:
%%writefile tier_alerts.py
"""Four-tier proton radiation alerts on the forward test period (2014 - Mar 2020).

  GREEN   quiet
  YELLOW  watch: storm likely within hours. Raised by observed flux >= 1 pfu, or (with --scores) by the Granite
          6 h forecast crossing its alarm threshold. Turns ON after 15 min of trigger (3 x 5-min readings, NOAA-style
          sustain rule) and OFF only after 2 h without trigger, so it does not flicker.
  ORANGE  storm: observed flux >= 10 pfu (NOAA S1-S2) - heightened monitoring, review EVA plans
  RED     EVA hazard: observed flux >= 1000 pfu (NOAA S3+, "radiation hazard avoidance recommended for astronauts on EVA")

Orange and red follow NOAA thresholds on observed flux; the model's job is the early YELLOW watch.
With --scores, also reports (exploratory) how early Granite's 24 h forecast reached 1000 pfu before each RED onset.

Usage:
  python tier_alerts.py OMNI.csv                                                   # observed-flux trigger only
  python tier_alerts.py OMNI.csv --scores out/granite_ttm_finetuned_scores.csv.gz  # Granite-raised yellow
Writes out/tier_episodes_<trigger>.csv and figs/fig4_tiers_<trigger>.png
"""
import sys, os, argparse
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.dates as mdates
from matplotlib.patches import Patch
import proton_forecast as pf

YEL, ORA, RED = 1.0, 10.0, 1000.0
ON_N, OFF_N = 3, 24                                   # 15 min on, 2 h off (5-min steps)
NAMES = ["green", "yellow", "orange", "red"]


def tiers(x, gflag=None):
    """0 green, 1 yellow, 2 orange, 3 red from 5-min flux x (NaN keeps previous state); gflag: Granite yellow trigger"""
    out = np.zeros(len(x), np.int8); on = False; up = down = 0
    for i, v in enumerate(x):
        trig = (not np.isnan(v) and v >= YEL) or (gflag is not None and gflag[i])
        if not np.isnan(v) or gflag is not None:
            up, down = (up + 1, 0) if trig else (0, down + 1)
            if not on and up >= ON_N: on = True
            if on and down >= OFF_N: on = False
        if not np.isnan(v) and v >= ORA:
            on = True; out[i] = 3 if v >= RED else 2
        else:
            out[i] = 1 if on else 0
    return out


def yellow_episodes(t, tr):
    """alerts starting from GREEN; escalated if they reach ORANGE; lead = start -> first ORANGE"""
    rows, n, i = [], len(tr), 0
    while i < n:
        if tr[i] == 1 and (i == 0 or tr[i - 1] == 0):
            j = i
            while j + 1 < n and tr[j + 1] != 0: j += 1
            seg = tr[i:j + 1]; o = np.flatnonzero(seg >= 2); r = np.flatnonzero(seg == 3)
            rows.append((t[i], t[j], len(o) > 0, (t[i + o[0]] - t[i]).total_seconds() / 3600 if len(o) else np.nan,
                         int(seg.max()), (t[i + r[0]] - t[i + o[0]]).total_seconds() / 3600 if len(r) else np.nan))
            i = j + 1
        else:
            i += 1
    return pd.DataFrame(rows, columns=["start", "end", "reached_orange", "lead_to_orange_h", "max_tier",
                                       "orange_to_red_h"])


def granite_flags(index, scores_path):
    S = pd.read_csv(scores_path, parse_dates=["time"]).set_index("time").sort_index()
    S = S.reindex(index, method="ffill", limit=2)               # 15-min forecasts held for the next two 5-min steps
    yflag = (S.max_lp_6h >= S.thr_6h).fillna(False).values
    redfc = (S.max_lp_24h >= np.log10(RED)).fillna(False).values
    return yflag, redfc, 10 ** S.thr_6h.dropna().iloc[0]


def red_forecast_check(t, x, redfc):
    """exploratory: for each RED onset, how early did the 24 h forecast first say >= 1000 pfu (within prior 48 h)?"""
    red = np.where(np.isnan(x), False, x >= RED)
    onset = [i for i in np.flatnonzero(red) if not red[max(0, i - 576):i].any()]     # no RED in prior 48 h
    rows = []
    for i in onset:
        w = np.flatnonzero(redfc[max(0, i - 576):i])
        rows.append((t[i], (t[i] - t[max(0, i - 576) + w[0]]).total_seconds() / 3600 if len(w) else np.nan))
    # forecast-RED episodes (gaps <= 1 h merged) not followed by an actual RED within 24 h
    f = np.flatnonzero(redfc); false_ = total = 0
    if len(f):
        brk = np.flatnonzero(np.diff(f) > 12)
        for a, b in zip(np.r_[f[0], f[brk + 1]], np.r_[f[brk], f[-1]]):
            total += 1
            false_ += not red[a:min(len(red), b + 288)].any()
    return pd.DataFrame(rows, columns=["red_onset", "forecast_lead_h"]), total, false_


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("csv"); ap.add_argument("--scores", default=None)
    ap.add_argument("--out", default="out"); ap.add_argument("--figs", default="figs")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True); os.makedirs(a.figs, exist_ok=True)
    trig = "granite" if a.scores else "observed"

    p = pf.load(a.csv).proton_flux_10mev
    test = p.loc["2014-01-01":]; x = test.values; t = test.index
    yflag = redfc = None
    if a.scores:
        yflag, redfc, thr_pfu = granite_flags(t, a.scores)
        print(f"YELLOW trigger: observed >= 1 pfu OR Granite 6 h forecast peak >= {thr_pfu:.2f} pfu")
    else:
        print("YELLOW trigger: observed flux >= 1 pfu")
    tr = tiers(x, yflag)
    valid = ~np.isnan(x); yrs = valid.sum() * 5 / 60 / 24 / 365.25
    print(f"test period: {yrs:.1f} years of valid data")
    for k, nm in enumerate(NAMES):
        h = ((tr == k) & valid).sum() * 5 / 60
        print(f"  {nm:6s} {h:7.0f} h  ({h / yrs:6.1f} h/yr, {100 * h / (valid.sum() * 5 / 60):.2f}% of time)")

    E = yellow_episodes(t, tr)
    esc, ne = E[E.reached_orange], E[~E.reached_orange]
    print(f"\nYELLOW alerts from quiet: {len(E)}  -> ORANGE: {len(esc)} ({100 * len(esc) / max(len(E), 1):.0f}%)  "
          f"-> no storm: {len(ne)} ({len(ne) / yrs:.1f}/yr)")
    if len(esc):
        print(f"  warning yellow->orange: median {esc.lead_to_orange_h.median():.1f} h, quartiles "
              f"{esc.lead_to_orange_h.quantile(.25):.1f}-{esc.lead_to_orange_h.quantile(.75):.1f} h")
    reds = E[E.max_tier == 3]
    for r in reds.itertuples():
        print(f"  RED (>=1000 pfu) in alert starting {r.start}: orange gave {r.orange_to_red_h:.1f} h before red")
    E.to_csv(f"{a.out}/tier_episodes_{trig}.csv", index=False)

    if redfc is not None:
        RF, tot, fal = red_forecast_check(t, x, redfc)
        print(f"\n[exploratory] Granite 24 h forecast reaching >=1000 pfu: {tot} forecast episodes, {fal} not followed by RED")
        for r in RF.itertuples():
            lead = "never forecast" if np.isnan(r.forecast_lead_h) else f"forecast {r.forecast_lead_h:.1f} h ahead"
            print(f"  RED onset {r.red_onset}: {lead}")
        print("  (only a handful of RED events: treat as anecdotal)")
    figure(p, E, trig, a)


def figure(p, E, trig, a):
    SURF, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e0"
    COL = {1: "#fab219", 2: "#ec835a", 3: "#d03b3b"}; ALPHA = {1: 0.24, 2: 0.16, 3: 0.42}; LINE = "#2a78d6"
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11, "axes.edgecolor": GRID, "axes.labelcolor": INK2,
                         "xtick.color": INK2, "ytick.color": INK2, "figure.facecolor": SURF, "axes.facecolor": SURF,
                         "savefig.facecolor": SURF, "axes.spines.top": False, "axes.spines.right": False})
    panels = [("Slow rise, S1–S2 storm: Sept 2014", "2014-09-09 12:00", "2014-09-12 12:00"),
              ("Fast rise, S3 storm: Sept 2017", "2017-09-09 12:00", "2017-09-12 12:00")]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.8), sharey=True)
    fig.subplots_adjust(left=0.07, right=0.87, top=0.72, bottom=0.14, wspace=0.08)
    src = "Granite forecast or observed flux" if trig == "granite" else "observed flux"
    fig.text(0.02, 0.965, "Four-tier radiation alerts", fontsize=15, fontweight="bold", color=INK, va="top")
    fig.text(0.02, 0.905, f"≥10 MeV proton flux. Yellow watch raised by {src}; orange and red follow NOAA S1 and S3 thresholds.",
             fontsize=10.5, color=INK2, va="top")
    yflag_all = None
    if trig == "granite":
        yflag_all, _, _ = granite_flags(p.index, a.scores)
    for ax, (ttl, s0, s1) in zip(axes, panels):
        i0, i1 = p.index.get_indexer([pd.Timestamp(s0), pd.Timestamp(s1)])
        lo = max(0, i0 - 2000)
        tr = tiers(p.values[lo:i1 + 1], None if yflag_all is None else yflag_all[lo:i1 + 1])[i0 - lo:]
        w = p.iloc[i0:i1 + 1]; tw = pd.Series(tr, index=w.index)
        for _, seg in tw.groupby((tw != tw.shift()).cumsum()):
            lvl = int(seg.iloc[0])
            if lvl: ax.axvspan(seg.index[0], seg.index[-1] + pd.Timedelta("5min"), color=COL[lvl], alpha=ALPHA[lvl], lw=0)
        ax.plot(w.index, w.values, color=LINE, lw=2)
        for y in (YEL, ORA, 100, RED):
            ax.axhline(y, color=INK2, lw=1, ls=(0, (4, 3)))
        ep = E[(E.start <= w.index[-1]) & (E.end >= w.index[0]) & E.reached_orange]
        if len(ep):
            o = ep.iloc[-1]; on = o.start + pd.Timedelta(hours=o.lead_to_orange_h)
            ax.annotate("", xy=(on, 0.15), xytext=(o.start, 0.15), arrowprops=dict(arrowstyle="<->", color=INK, lw=1.2))
            ax.text(o.start - pd.Timedelta("2h"), 0.15, f"{o.lead_to_orange_h:.1f} h watch ", ha="right", va="center",
                    fontsize=10.5, color=INK)
        ax.set_title(ttl, loc="left", fontsize=12, color=INK, pad=8)
        ax.set_yscale("log"); ax.set_ylim(0.1, 5000)
        ax.xaxis.set_major_locator(mdates.DayLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    axes[0].set_ylabel("Proton flux ≥10 MeV (pfu, log)")
    for y, lab in [(YEL, "1 pfu"), (ORA, "10 pfu  S1"), (100, "100 pfu  S2"), (RED, "1000 pfu  S3")]:
        axes[1].text(1.02, y, lab, transform=axes[1].get_yaxis_transform(), va="center", fontsize=9.5, color=INK2)
    axes[0].legend(handles=[Patch(color=COL[1], alpha=0.5, label="YELLOW: watch"),
                            Patch(color=COL[2], alpha=0.4, label="ORANGE: storm, 10–1000 pfu"),
                            Patch(color=COL[3], alpha=0.4, label="RED: EVA hazard, ≥1000 pfu")],
                   frameon=False, loc="upper left", fontsize=10, ncol=3, bbox_to_anchor=(0, 1.24))
    fig.text(0.02, 0.03, "Yellow turns on after 15 min of trigger and off after 2 h without. Arrow: watch time before the storm (orange).",
             fontsize=9, color=INK2)
    out = f"{a.figs}/fig4_tiers_{trig}.png"
    fig.savefig(out, dpi=200); print(f"\nfigure -> {out}")


if __name__ == "__main__":
    main()

In [ ]:
!python tier_alerts.py "{CSV}"                                                    # observed-flux trigger

In [ ]:
!python tier_alerts.py "{CSV}" --scores out/granite_ttm_finetuned_scores.csv.gz   # Granite-raised yellow

In [ ]:
from IPython.display import Image, display
for f in ["figs/fig4_tiers_observed.png", "figs/fig4_tiers_granite.png"]:
    if os.path.exists(f): display(Image(f, width=1000))